# CMIP6-decadal recipe example

Shows the standalone Woodpecker calls used for C3S CMIP6-decadal adaptation.

Flow: create synthetic decadal data -> load `c3s.cmip6_decadal` -> run `prepare` -> run `apply` -> re-check.

In [ ]:
import numpy as np
import woodpecker_cmip6_decadal_plugin  # noqa: F401 - imports plugin fixes for editable installs
import xarray as xr

import woodpecker
from woodpecker.recipe import APPLY_PHASE, PREPARE_PHASE
from woodpecker.testing import make_cmip6_decadal

Create a tiny CMIP6-decadal-like dataset with changes that exercise both phases.

The calendar encoding is a prepare-phase change. The remaining metadata and coordinate changes are normal apply-phase adaptations.

In [ ]:
source_name = (
    "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3."
    "dcppA-hindcast.s1960-r2i1p1f1.Amon.tas.gr.v20201215.nc"
)

dataset = make_cmip6_decadal(
    overrides={
        "source_name": source_name,
        "startdate": "s1960",
        "sub_experiment_id": "s1960",
        "realization_index": "2",
        "forcing_description": "wrong",
    }
)
dataset = dataset.isel(time=slice(0, 2))
dataset = dataset.assign_coords(time=np.array(["1960-11-16", "1960-12-16"], dtype="datetime64[D]"))
dataset["time"].attrs["long_name"] = "time"
dataset["time"].encoding["calendar"] = "proleptic_gregorian"
dataset["realization"] = xr.DataArray(
    2,
    attrs={"comment": "short", "long_name": "member"},
)
dataset["realization"].encoding["_FillValue"] = -9999

dataset

Load the bundled recipe and inspect the phase layout.

In [ ]:
recipe = woodpecker.recipe.get("c3s.cmip6_decadal")

[(step.phase, step.id) for step in recipe.steps]

Run the pre-concatenation preparation step explicitly with `phase=PREPARE_PHASE`.

In [ ]:
prepare_findings = woodpecker.recipe.check(dataset, recipe, phase=PREPARE_PHASE)
prepare_preview = woodpecker.recipe.apply(dataset, recipe, phase=PREPARE_PHASE, dry_run=True)
prepare_write = woodpecker.recipe.apply(dataset, recipe, phase=PREPARE_PHASE, dry_run=False)

(
    prepare_findings.fix_ids,
    prepare_preview.changed,
    prepare_write.changed,
    dataset["time"].encoding["calendar"],
)

The normal C3S/CDS adaptation uses the same public API with `phase=APPLY_PHASE`.

In [ ]:
apply_findings = woodpecker.recipe.check(dataset, recipe, phase=APPLY_PHASE)
apply_preview = woodpecker.recipe.apply(dataset, recipe, phase=APPLY_PHASE, dry_run=True)
apply_write = woodpecker.recipe.apply(dataset, recipe, phase=APPLY_PHASE, dry_run=False)

(
    apply_findings.fix_ids,
    apply_preview.changed,
    apply_write.changed,
)

After both phases, the synthetic dataset has the expected C3S-ready fields.

In [ ]:
(
    dataset.attrs["startdate"],
    dataset.attrs["sub_experiment_id"],
    dataset.attrs["forcing_description"],
    dataset.attrs["realization_index"],
    dataset["time"].attrs["long_name"],
    dataset["time"].encoding["calendar"],
    dataset["realization"].dtype,
    dataset["realization"].attrs["long_name"],
    "_FillValue" in dataset["realization"].encoding,
    "reftime" in dataset.coords,
    "leadtime" in dataset.coords,
)

In [ ]:
assert dataset.attrs["startdate"] == "s196011"
assert dataset.attrs["sub_experiment_id"] == "s196011"
assert dataset.attrs["forcing_description"] == "f1, CMIP6 historical forcings"
assert dataset.attrs["realization_index"] == 2
assert dataset["time"].attrs["long_name"] == "valid_time"
assert dataset["time"].encoding["calendar"] == "standard"
assert dataset["realization"].dtype == np.int32
assert dataset["realization"].attrs["long_name"] == "realization"
assert "_FillValue" not in dataset["realization"].encoding
assert "reftime" in dataset.coords
assert "leadtime" in dataset.coords

bool(woodpecker.recipe.check(dataset, recipe))